# StreamForge — FLUX.1-dev in Pure Rust on Kaggle

Full BF16 FLUX.1-dev inference pipeline written in **pure Rust** using [candle](https://github.com/huggingface/candle).  
Streams the 24GB transformer **block-by-block** from disk through CPU RAM to GPU — peak VRAM ~4GB.

**Source code:** https://github.com/madtunebk/GPULoad

## What this notebook does
1. Mounts the pre-converted model dataset (flux_candle + vae_candle safetensors)
2. Downloads CLIP-L + T5-XXL weights from HuggingFace (~10GB, ~2 min)
3. Uses pre-built binaries OR builds from source if glibc mismatch
4. Runs the full pipeline: text_encoder → flux_gpu → vae_decode → PNG

## Required Kaggle Dataset
Add your dataset (containing `flux_candle.safetensors` and `vae_candle.safetensors`) via:  
**Add Data → Your Datasets → streamforge-models**

It will be mounted at `/kaggle/input/streamforge-models/`

## GPU
Enable GPU accelerator: **Settings → Accelerator → GPU T4 x2** (or P100)

In [ ]:
# Verify GPU is available
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], 
                        capture_output=True, text=True)
print(result.stdout)

import os
print(f"HOME={os.environ.get('HOME')}")
print(f"CUDA available:", os.path.exists('/usr/local/cuda'))

## Step 1 — Check model dataset is mounted

## Step 1b — HuggingFace Authentication

FLUX.1-dev is a **gated model** — you must:
1. Accept the license at https://huggingface.co/black-forest-labs/FLUX.1-dev
2. Create a **classic** Read token at https://huggingface.co/settings/tokens  
   ⚠️ Fine-grained tokens block gated repos by default — use classic OR enable "Access to public gated repositories" in the fine-grained token settings
3. Add it as a Kaggle Secret: **Add-ons → Secrets → Add** with name `HF_TOKEN`

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token  # older HF versions
print("HF token loaded")

In [ ]:
import os

DATASET_DIR = "/kaggle/input/streamforge-models"
FLUX_MODEL  = f"{DATASET_DIR}/flux_candle.safetensors"
VAE_MODEL   = f"{DATASET_DIR}/vae_candle.safetensors"

for path in [FLUX_MODEL, VAE_MODEL]:
    size = os.path.getsize(path) / 1e9
    print(f"OK  {path}  ({size:.1f} GB)")

## Step 2 — Download CLIP-L + T5-XXL from HuggingFace

These weights stay in the HF cache at `~/.cache/huggingface/` which matches the hardcoded paths in the Rust binaries.

> Takes ~2 minutes on Kaggle's network.

In [ ]:
!pip install -q huggingface_hub

# Download CLIP + T5 + tokenizers only (transformer comes from the Kaggle dataset)
from huggingface_hub import snapshot_download
import os

snapshot_download(
    repo_id="black-forest-labs/FLUX.1-dev",
    allow_patterns=["text_encoder/**", "text_encoder_2/**", "tokenizer/**", "tokenizer_2/**"],
    token=os.environ["HF_TOKEN"],
    ignore_patterns=["*.bin"],   # skip pytorch_model.bin if present, keep safetensors only
)

print("Done.")

## Step 3 — Set up workspace and symlink models

In [ ]:
import os, subprocess

WORKDIR = "/kaggle/working/streamforge"
os.makedirs(f"{WORKDIR}/models", exist_ok=True)
os.makedirs(f"{WORKDIR}/temp",   exist_ok=True)
os.makedirs(f"{WORKDIR}/loras",  exist_ok=True)

# Symlink pre-converted models into the relative path the binaries expect
for fname, src in [("flux_candle.safetensors", FLUX_MODEL), ("vae_candle.safetensors", VAE_MODEL)]:
    dst = f"{WORKDIR}/models/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    print(f"linked {dst}")

# Symlink tokenizer
tok_src = os.path.expanduser(
    "~/.cache/huggingface/hub/models--black-forest-labs--FLUX.1-dev"
)
# Find the snapshot dir dynamically (hash may differ)
snaps = [d for d in os.listdir(f"{tok_src}/snapshots")]
SNAP = f"{tok_src}/snapshots/{snaps[0]}"
print(f"Snapshot: {SNAP}")

tok_dst = f"{WORKDIR}/models/tokenizer"
if not os.path.exists(tok_dst):
    os.symlink(f"{SNAP}/tokenizer", tok_dst)
print(f"linked {tok_dst}")

## Step 4 — Get pre-built binaries (or build from source)

Try the pre-built binaries first. If they fail due to glibc mismatch, fall back to building from source (~4 min).

In [ ]:
import os, subprocess, shutil

BINARIES_DIR = "/kaggle/input/streamforge-models/binaries"
BIN_DST      = f"{WORKDIR}/bin"
os.makedirs(BIN_DST, exist_ok=True)

BINS = ["text_encoder", "flux_gpu", "vae_decode"]
use_prebuilt = all(os.path.exists(f"{BINARIES_DIR}/{b}") for b in BINS)

if use_prebuilt:
    print("Pre-built binaries found — copying...")
    for b in BINS:
        dst = f"{BIN_DST}/{b}"
        shutil.copy2(f"{BINARIES_DIR}/{b}", dst)
        os.chmod(dst, 0o755)
    # Quick smoke test
    r = subprocess.run([f"{BIN_DST}/flux_gpu", "--help"], capture_output=True, text=True)
    if r.returncode != 0 and "GLIBC" in (r.stderr or ""):
        print("glibc mismatch — falling back to source build")
        use_prebuilt = False
    else:
        print("Binaries OK")

if not use_prebuilt:
    print("Building from source — this takes ~4 minutes...")
    # Install Rust
    subprocess.run('curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y', 
                   shell=True, check=True)
    os.environ["PATH"] = f"/root/.cargo/bin:{os.environ['PATH']}"
    
    # Clone repo
    repo_dir = "/kaggle/working/GPULOAD"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "https://github.com/madtunebk/GPULoad.git", repo_dir], check=True)
    
    subprocess.run(["cargo", "build", "--release", "--features", "cuda"],
                   cwd=repo_dir, check=True)
    
    for b in BINS:
        dst = f"{BIN_DST}/{b}"
        shutil.copy2(f"{repo_dir}/target/release/{b}", dst)
        os.chmod(dst, 0o755)
    print("Build complete")

## Step 5 — Run the pipeline

Edit `PROMPT`, `WIDTH`, `HEIGHT`, `STEPS` below.

In [ ]:
PROMPT   = "anime2, masterpiece, best quality, 1girl, white hair, red eyes, holding a glowing blue sword, fantasy armor, epic background"
WIDTH    = 768
HEIGHT   = 1024
STEPS    = 20
GUIDANCE = 3.5
SEED     = 42
LORA     = None   # e.g. "/kaggle/input/streamforge-models/loras/my.safetensors"
LORA_SCALE = 1.0

In [ ]:
import subprocess, os, time

env = os.environ.copy()
env["PATH"] = f"{BIN_DST}:{env['PATH']}"

def run(cmd, **kwargs):
    print(f"$ {' '.join(cmd)}")
    t0 = time.time()
    result = subprocess.run(cmd, cwd=WORKDIR, env=env, **kwargs)
    print(f"  → {time.time()-t0:.1f}s")
    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")

# --- 1. text_encoder ---
te_cmd = [f"{BIN_DST}/text_encoder", PROMPT, "--device", "cuda"]
if LORA:
    te_cmd += ["--lora", LORA, "--lora-scale", str(LORA_SCALE)]
run(te_cmd)

# --- 2. flux_gpu ---
fg_cmd = [f"{BIN_DST}/flux_gpu",
          "--width",    str(WIDTH),
          "--height",   str(HEIGHT),
          "--steps",    str(STEPS),
          "--guidance", str(GUIDANCE),
          "--seed",     str(SEED)]
if LORA:
    fg_cmd += ["--lora", LORA, "--lora-scale", str(LORA_SCALE)]
run(fg_cmd)

# --- 3. vae_decode ---
run([f"{BIN_DST}/vae_decode"])

## Result

In [ ]:
from IPython.display import Image, display
import shutil, os

output_src = f"{WORKDIR}/temp/output_rust.png"
output_dst = "/kaggle/working/output_rust.png"
shutil.copy2(output_src, output_dst)

display(Image(filename=output_dst))
print(f"Saved to {output_dst}")